# ARC_ATLAS v4 Slice-Block Conservative Resume

This notebook resumes from the best stable slice-block checkpoint and uses a conservative fine-tune. The previous resume run failed scientifically because the loss made false positives too cheap: predicted lesion volume ballooned, the best validation threshold pinned at `0.50`, and whole-brain Dice fell after epoch 0.

This version keeps the architecture unchanged and starts from:
`runs/20260420_120656_slice_blocks/callbacks/best_slice_block.weights.h5`

Changes here are deliberately mild:
- lower LR: `1e-5`
- mild recall tilt: Tversky `alpha=0.60`, `beta=0.40`
- empty slices still contribute to Dice/Tversky
- lesion slices are only lightly upweighted
- positive focus term is small
- decision threshold starts at `0.15`, matching the best original run neighborhood
- threshold sweep includes both lower and higher thresholds
- early stopping watches true whole-brain Dice and stops if it stalls
- target stop is set to `0.80`, but that is a target, not a guarantee


In [1]:
from pathlib import Path
import sys
import time
import importlib

candidates = [
    Path.cwd(),
    Path.cwd() / "ARC_ATLAS_Combined" / "ARC_ATLAS_Train_v4",
    Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4"),
]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "src" / "training_v2_slice_blocks.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate ARC_ATLAS_Train_v4/src/training_v2_slice_blocks.py")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import training_v2_slice_blocks as seg
seg = importlib.reload(seg)

TRAIN_DIR = PROJECT_ROOT / "data" / "splits" / "90_10_random" / "train"
SOURCE_RUN = PROJECT_ROOT / "runs" / "20260420_120656_slice_blocks"
INITIAL_WEIGHTS = SOURCE_RUN / "callbacks" / "best_slice_block.weights.h5"
RUN_DIR = PROJECT_ROOT / "runs" / f"{time.strftime('%Y%m%d_%H%M%S')}_slice_blocks_resume_conservative"

if not INITIAL_WEIGHTS.exists():
    raise FileNotFoundError(f"Resume checkpoint not found: {INITIAL_WEIGHTS}")

cfg = seg.SliceBlockTrainingConfig(
    DATA_DIR=TRAIN_DIR,
    IMAGES_DIR=TRAIN_DIR / "t1",
    MASKS_DIR=TRAIN_DIR / "masks",
    MANIFEST_PATH=TRAIN_DIR / "manifest.csv",
    MODEL_DIR=RUN_DIR / "models",
    CALLBACKS_DIR=RUN_DIR / "callbacks",
    INITIAL_WEIGHTS_PATH=INITIAL_WEIGHTS,
    TARGET_SHAPE=(192, 224, 192),
    RESAMPLE_TO_TARGET=False,
    SLICE_AXIS=2,
    BLOCK_DEPTH=3,
    SLICE_STRIDE=1,
    TOTAL_EPOCHS=24,
    INITIAL_LR=1e-5,
    MIN_LR=2e-7,
    MIXED_PRECISION=False,
    JIT_COMPILE=False,
    BASE_FILTERS=8,
    UNET_DEPTH=4,
    DROPOUT_RATE=0.10,
    L2_REG=1e-4,
    POSITIVE_WEIGHT=38.0,
    BCE_WEIGHT=0.46,
    DICE_WEIGHT=0.44,
    FOCAL_TVERSKY_WEIGHT=0.10,
    TVERSKY_ALPHA=0.60,
    TVERSKY_BETA=0.40,
    FOCAL_TVERSKY_GAMMA=1.33,
    LESION_SLICE_WEIGHT=1.25,
    EMPTY_SLICE_WEIGHT=0.85,
    DICE_ON_LESION_SLICES_ONLY=False,
    POSITIVE_TOPK_WEIGHT=0.015,
    POSITIVE_TOPK_FRACTION=0.25,
    AUGMENT=True,
    AUG_INTENSITY_SCALE=0.05,
    AUG_INTENSITY_SHIFT=0.02,
    AUG_NOISE_STD=0.005,
    DECISION_THRESHOLD=0.15,
    VAL_THRESHOLD_SWEEP=(0.05, 0.075, 0.10, 0.125, 0.15, 0.175, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70),
    WHOLE_BRAIN_VAL_EVERY_N_EPOCHS=1,
    WHOLE_BRAIN_VAL_MAX_CASES=None,
    EARLY_STOPPING_PATIENCE=5,
    EARLY_STOPPING_MIN_DELTA=0.001,
    RESTORE_BEST_WEIGHTS=True,
    TARGET_WHOLE_DICE=0.80,
    SAVE_VAL_PREDICTIONS=True,
    NUM_VAL_PREDICTIONS=5,
    FIT_VERBOSE=2,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Resume checkpoint: {INITIAL_WEIGHTS}")
print(f"New run dir: {RUN_DIR}")
print(f"Input per brain: (num_slices, {cfg.input_shape[0]}, {cfg.input_shape[1]}, {cfg.input_shape[2]})")
print(f"Decision threshold: {cfg.DECISION_THRESHOLD}")
print(f"Threshold sweep: {cfg.VAL_THRESHOLD_SWEEP}")
print(f"Early stopping patience: {cfg.EARLY_STOPPING_PATIENCE}")
print(f"Target whole-brain Dice: {cfg.TARGET_WHOLE_DICE}")
print(f"Epoch log CSV: {cfg.CALLBACKS_DIR / 'training_log.csv'}")
print(f"Whole-brain validation summary: {cfg.CALLBACKS_DIR / 'whole_val_summary.jsonl'}")


2026-04-22 10:37:31.756090: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Project root: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4
Resume checkpoint: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260420_120656_slice_blocks/callbacks/best_slice_block.weights.h5
New run dir: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260422_103733_slice_blocks_resume_conservative
Input per brain: (num_slices, 192, 224, 3)
Decision threshold: 0.15
Threshold sweep: (0.05, 0.075, 0.1, 0.125, 0.15, 0.175, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7)
Early stopping patience: 5
Target whole-brain Dice: 0.8
Epoch log CSV: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260422_103733_slice_blocks_resume_conservative/callbacks/training_log.csv
Whole-brain validation summary: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260422_103733_slice_blocks_resume_conservative/callbacks/whole_val_summary.jsonl


In [2]:
# Verify the checkpoint loads before starting the fine-tune.
model = seg.build_slice_block_model(cfg)
model.load_weights(str(INITIAL_WEIGHTS))
print(f"Loaded weights into model with {model.count_params():,} parameters")
del model
seg.tf.keras.backend.clear_session()


I0000 00:00:1776875853.892428  280915 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1776875853.893861  280915 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22122 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9


Loaded weights into model with 493,345 parameters


In [3]:
# Sanity check one full-geometry case under the resume config.
cases = seg.load_cases(cfg)
image, mask, _ = seg.load_case_arrays(cases[0], cfg)
x, y = seg.make_slice_blocks(image, mask, cfg)
print(f"Cases: {len(cases)}")
print(f"Prepared brain: image={image.shape}, mask={mask.shape}")
print(f"One-brain batch: x={x.shape}, y={y.shape}")
print(f"Lesion voxels in sanity case: {int(y.sum())}")


2026-04-22 10:37:34,955 - SliceBlockTrainer - INFO - Using manifest: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/manifest.csv
2026-04-22 10:37:34,975 - SliceBlockTrainer - INFO - Loaded 866 cases from manifest


Cases: 866
Prepared brain: image=(192, 224, 192), mask=(192, 224, 192)
One-brain batch: x=(192, 192, 224, 3), y=(192, 192, 224, 1)
Lesion voxels in sanity case: 149459


In [4]:
# Launch conservative resume fine-tune.
# Epoch numbers here are phase-local; epoch 0 means first fine-tune epoch after loading best weights.
history = seg.train_slice_block_model(cfg)


2026-04-22 10:37:35,515 - SliceBlockTrainer - WARNING - Could not set memory growth on PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'): Physical devices cannot be modified after being initialized
2026-04-22 10:37:35,516 - SliceBlockTrainer - WARNING - Could not set memory growth on PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'): Physical devices cannot be modified after being initialized
2026-04-22 10:37:35,517 - SliceBlockTrainer - INFO - Using manifest: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/manifest.csv
2026-04-22 10:37:35,538 - SliceBlockTrainer - INFO - Loaded 866 cases from manifest
2026-04-22 10:39:41,836 - SliceBlockTrainer - INFO - Split cases: train=737 val=129
2026-04-22 10:39:42,039 - SliceBlockTrainer - INFO - Loading initial weights from /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260420_120656_slice_blocks/callbacks/best_slice_block.weights.h5
202

Epoch 1/24


E0000 00:00:1776875987.640839  280915 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/SliceBlock2p5D_UNet_1/enc1_dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2026-04-22 10:39:49.343758: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91001
2026-04-22 10:53:24,150 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 16/129
2026-04-22 10:53:48,794 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 32/129
2026-04-22 10:54:12,783 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 48/129
2026-04-22 10:54:36,911 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 64/129
2026-04-22 10:55:00,955 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 80/129
2026-04-22 10:55:25,014 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 96/129
2026


Epoch 1: val_whole_dice_hard_best_thr_score improved from None to 0.39501, saving model to /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260422_103733_slice_blocks_resume_conservative/callbacks/best_slice_block.weights.h5
737/737 - 1002s - 1s/step - dice_coefficient: 0.0931 - foreground_fraction: 0.0209 - hard_dice_metric: 0.7165 - loss: 0.5732 - val_dice_coefficient: 0.0965 - val_foreground_fraction: 0.0053 - val_hard_dice_metric: 0.8451 - val_loss: 0.6242 - val_whole_dice_soft_macro: 0.2872 - val_whole_dice_hard: 0.3386 - val_whole_dice_hard_best_thr: 0.5000 - val_whole_dice_hard_best_thr_score: 0.3950
Epoch 2/24


2026-04-22 11:09:52,224 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 16/129
2026-04-22 11:10:16,642 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 32/129
2026-04-22 11:10:41,089 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 48/129
2026-04-22 11:11:05,497 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 64/129
2026-04-22 11:11:29,947 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 80/129
2026-04-22 11:11:54,455 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 96/129
2026-04-22 11:12:19,038 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 112/129
2026-04-22 11:12:43,476 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 128/129
2026-04-22 11:12:45,034 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 129/129
2026-04-22 11:12:45,038 - SliceBlockTrainer - INFO - Whole-brain slice-block val @epoch 1: soft=0.28841 hard@0.15=0

...


Epoch 4: val_whole_dice_hard_best_thr_score did not improve from 0.39501
737/737 - 979s - 1s/step - dice_coefficient: 0.0919 - foreground_fraction: 0.0213 - hard_dice_metric: 0.6860 - loss: 0.5687 - val_dice_coefficient: 0.0969 - val_foreground_fraction: 0.0060 - val_hard_dice_metric: 0.8331 - val_loss: 0.6043 - val_whole_dice_soft_macro: 0.2719 - val_whole_dice_hard: 0.3153 - val_whole_dice_hard_best_thr: 0.7000 - val_whole_dice_hard_best_thr_score: 0.3882
Epoch 5/24


2026-04-22 11:58:51,374 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 16/129
2026-04-22 11:59:15,931 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 32/129
2026-04-22 11:59:40,453 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 48/129
2026-04-22 12:00:04,744 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 64/129
2026-04-22 12:00:29,324 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 80/129
2026-04-22 12:00:54,002 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 96/129
2026-04-22 12:01:18,411 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 112/129
2026-04-22 12:01:42,854 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 128/129
2026-04-22 12:01:44,378 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 129/129
2026-04-22 12:01:44,383 - SliceBlockTrainer - INFO - Whole-brain slice-block val @epoch 4: soft=0.27761 hard@0.15=0


Epoch 5: val_whole_dice_hard_best_thr_score did not improve from 0.39501
737/737 - 980s - 1s/step - dice_coefficient: 0.0933 - foreground_fraction: 0.0211 - hard_dice_metric: 0.6836 - loss: 0.5688 - val_dice_coefficient: 0.0968 - val_foreground_fraction: 0.0059 - val_hard_dice_metric: 0.8312 - val_loss: 0.6094 - val_whole_dice_soft_macro: 0.2776 - val_whole_dice_hard: 0.3136 - val_whole_dice_hard_best_thr: 0.7000 - val_whole_dice_hard_best_thr_score: 0.3865
Epoch 6/24


2026-04-22 12:15:11,846 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 16/129
2026-04-22 12:15:36,212 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 32/129
2026-04-22 12:16:00,764 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 48/129
2026-04-22 12:16:25,146 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 64/129
2026-04-22 12:16:49,703 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 80/129
2026-04-22 12:17:14,226 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 96/129
2026-04-22 12:17:39,038 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 112/129
2026-04-22 12:18:03,655 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 128/129
2026-04-22 12:18:05,178 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 129/129
2026-04-22 12:18:05,182 - SliceBlockTrainer - INFO - Whole-brain slice-block val @epoch 5: soft=0.26184 hard@0.15=0


Epoch 6: val_whole_dice_hard_best_thr_score did not improve from 0.39501
737/737 - 981s - 1s/step - dice_coefficient: 0.0926 - foreground_fraction: 0.0212 - hard_dice_metric: 0.6767 - loss: 0.5668 - val_dice_coefficient: 0.0946 - val_foreground_fraction: 0.0055 - val_hard_dice_metric: 0.8347 - val_loss: 0.6095 - val_whole_dice_soft_macro: 0.2618 - val_whole_dice_hard: 0.3125 - val_whole_dice_hard_best_thr: 0.6000 - val_whole_dice_hard_best_thr_score: 0.3912
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 1.


In [ ]:
# Review the fine-tune logs.
import json
import pandas as pd

train_log = cfg.CALLBACKS_DIR / "training_log.csv"
whole_val_log = cfg.CALLBACKS_DIR / "whole_val_summary.jsonl"

if train_log.exists():
    display(pd.read_csv(train_log).tail(10))
else:
    print(f"Training log not found yet: {train_log}")

if whole_val_log.exists():
    rows = [json.loads(line) for line in whole_val_log.read_text().splitlines() if line.strip()]
    df = pd.DataFrame(rows)
    display(df.tail(10))
    display(df.sort_values("val_whole_dice_hard_best_thr_score", ascending=False).head(10))
else:
    print(f"Whole-brain validation log not found yet: {whole_val_log}")


,epoch,dice_coefficient,foreground_fraction,hard_dice_metric,loss,val_dice_coefficient,val_foreground_fraction,val_hard_dice_metric,val_loss,val_whole_dice_hard,val_whole_dice_hard_best_thr,val_whole_dice_hard_best_thr_score,val_whole_dice_soft_macro
0,0,0.093062,0.020941,0.716478,0.573178,0.096491,0.005288,0.845145,0.624231,0.338646,0.5,0.395008,0.287236
1,1,0.092611,0.021001,0.703330,0.570250,0.098247,0.005997,0.837009,0.613352,0.325091,0.7,0.389196,0.288410
2,2,0.092360,0.021120,0.691740,0.569107,0.096360,0.005522,0.840369,0.613699,0.325748,0.6,0.393037,0.275056
3,3,0.091932,0.021292,0.685982,0.568715,0.096919,0.006004,0.833086,0.604289,0.315270,0.7,0.388192,0.271903
4,4,0.093285,0.021125,0.683632,0.568772,0.096794,0.005900,0.831198,0.609372,0.313609,0.7,0.386518,0.277615
5,5,0.092563,0.021194,0.676662,0.566761,0.094634,0.005541,0.834696,0.609538,0.312518,0.6,0.391215,0.261844


,epoch,elapsed_sec,n_cases,val_whole_dice_soft_macro,val_whole_dice_hard,val_whole_dice_hard_best_thr,val_whole_dice_hard_best_thr_score,pred_max_p90,pred_hard_voxels_median
0,0,195.708812,129,0.287236,0.338646,0.5,0.395008,0.998562,26286.0
1,1,197.521240,129,0.288410,0.325091,0.7,0.389196,0.998722,30288.0
2,2,197.797001,129,0.275056,0.325748,0.6,0.393037,0.998467,30972.0
3,3,197.735502,129,0.271903,0.315270,0.7,0.388192,0.998438,34263.0
4,4,197.826278,129,0.277615,0.313609,0.7,0.386518,0.998528,35300.0
5,5,197.989956,129,0.261844,0.312518,0.6,0.391215,0.998031,34951.0


,epoch,elapsed_sec,n_cases,val_whole_dice_soft_macro,val_whole_dice_hard,val_whole_dice_hard_best_thr,val_whole_dice_hard_best_thr_score,pred_max_p90,pred_hard_voxels_median
0,0,195.708812,129,0.287236,0.338646,0.5,0.395008,0.998562,26286.0
2,2,197.797001,129,0.275056,0.325748,0.6,0.393037,0.998467,30972.0
5,5,197.989956,129,0.261844,0.312518,0.6,0.391215,0.998031,34951.0
1,1,197.521240,129,0.288410,0.325091,0.7,0.389196,0.998722,30288.0
3,3,197.735502,129,0.271903,0.315270,0.7,0.388192,0.998438,34263.0
4,4,197.826278,129,0.277615,0.313609,0.7,0.386518,0.998528,35300.0


: 